# BirdSense — Colab Training Notebook

**Before running:**
1. Set runtime type to **GPU** — Runtime > Change runtime type > T4 GPU
2. Upload your project to Google Drive so the layout below matches
3. Run all cells top-to-bottom

**Expected Drive layout:**
```
MyDrive/
└── birdsense/
    ├── dataset/
    │   ├── processed/      <- upload this (8475 wav clips)
    │   └── test_holdout/   <- upload this (holdout mp3s)
    ├── scripts/            <- upload all .py files
    ├── species_selected.csv
    └── requirements_train.txt
```

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2 — Set Project Path

Edit `PROJECT_DIR` if your folder is named differently.

In [ ]:
import os

PROJECT_DIR = '/content/drive/MyDrive/birdsense'
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())

for folder in ['dataset/processed', 'dataset/test_holdout', 'scripts']:
    status = 'OK' if os.path.isdir(folder) else 'MISSING'
    print(f'  [{status}]  {folder}')

## Step 3 — Install Dependencies

In [ ]:
!pip install -q -r requirements_train.txt

## Step 4 — Check GPU

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU available: {gpus}')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.')

## Step 5 — Check Dataset

Verify clip counts per species before training.

In [ ]:
from pathlib import Path

processed = Path('dataset/processed')
holdout = Path('dataset/test_holdout')

total_clips = 0
print(f'{"Species":<35} {"Processed":>10} {"Holdout":>10}')
print('-' * 57)
for species_dir in sorted(processed.iterdir()):
    if not species_dir.is_dir():
        continue
    clips = len(list(species_dir.glob('*.wav')))
    holdout_dir = holdout / species_dir.name
    ho = len(list(holdout_dir.glob('*.mp3'))) if holdout_dir.exists() else 0
    total_clips += clips
    print(f'{species_dir.name:<35} {clips:>10} {ho:>10}')
print('-' * 57)
print(f'{"TOTAL":<35} {total_clips:>10}')

## Step 6 — Train

- Downloads YAMNet from TF Hub once (~25 MB, then cached)
- Extracts embeddings for all clips (~10–15 min first run, instant on reruns)
- Trains Dense head for up to 30 epochs with early stopping
- Saves best checkpoint to `models/checkpoints/best_head.keras`

In [ ]:
!python scripts/train.py

## Step 7 — Evaluate

Runs the trained model on the held-out test set.
Checks macro-F1 against the 0.80 export gate.
Report saved to `models/checkpoints/eval_report.txt`.

In [ ]:
!python scripts/evaluate.py

## Step 8 — Export

**Only run if Step 7 printed PASS.**

Saves:
- `models/export/model.tflite`
- `models/export/labels.txt`

In [ ]:
!python scripts/export.py

## Step 9 — Download Outputs

Downloads the exported files to your local machine.

In [ ]:
from google.colab import files
import os

outputs = [
    'models/export/model.tflite',
    'models/export/labels.txt',
    'models/checkpoints/eval_report.txt',
    'models/checkpoints/label_map.json',
]

for path in outputs:
    if os.path.exists(path):
        print(f'Downloading: {path}')
        files.download(path)
    else:
        print(f'Not found (skipping): {path}')